# [2장 2강] - 실습: 부스팅 계열 모델의 원리와 발전

## 실습 배경
- 와인 품종을 분류하는 여러 모델을 비교해, 다음 배포 후보 하나를 정한다.
- 데이터에 기존에 있던 연속 측정값을 구간으로 나눠 만든 "범주 코드"를 추가한다.
- 예를 들어 알코올 도수를 낮음/중간/높음으로 잘라 0, 1, 2로 표시한 열이다.
- 이 값들은 숫자처럼 보이지만 실제로는 범주이므로, 전처리 단계에서 "이 열은 연속값이 아니라 범주"라고 명확히 구분해 처리해야 한다.
- 세 모델(배깅 1개 + 부스팅 2개)을 동일한 교차검증 설정(같은 fold 분할)으로 비교한다.
- 이어서 부스팅의 학습률(learning_rate)과 반복 수(n_estimators)가 성능에 어떤 영향을 주는지 살펴본 뒤 최종 후보를 결정한다.
- 최종 결정 기준은 개발 데이터의 평균 AP다.
- Early Stopping 결과는 "나무를 몇 개나 쓰는 게 적절한가"를 진단하는 별도 실험으로만 쓴다.
- Early Stopping은 검증 절차가 일반 교차검증과 다르므로, 두 절차에서 나온 점수를 한 표에 섞어 순위를 매기지 않는다.

## 실습 목표
- 숫자로 표현된 범주 열과 연속형 특성을 구분해서, 각각 다른 전처리를 적용하도록 `ColumnTransformer`를 구성한다. (범주 코드는 범주 전처리로, 연속값은 연속값 전처리로)
- 배깅 1개(Random Forest)와 부스팅 2개(AdaBoost, Gradient Boosting)를 동일한 `StratifiedKFold`(같은 fold 분할)로 공정하게 비교한다.
- 부스팅의 `learning_rate`와 `n_estimators` 조합을 바꿔가며, 평균 AP와 fold 간 변동(표준편차)이 어떻게 달라지는지 해석한다.
- Early Stopping을 적용해 (1) 학습이 실제로 몇 번째 반복에서 멈췄는지와 (2) 그때의 validation 성능을 확인한다.
  → 이건 "적정 반복 수를 진단하는 별도 실험"이다.
- 모든 후보를 같은 방식(같은 dev 데이터·같은 CV·같은 AP 지표)으로 평가한 결과 중에서 최종 모델을 하나 고르고, test는 마지막에 딱 한 번만 평가한다. (Early Stopping처럼 평가 방식이 다른 점수는 이 선택에 섞지 않는다)

## 입력 데이터 카드

- **출처**: scikit-learn Wine recognition 데이터

- **크기**: `X.shape == (178, 15)`
  - 원본 연속 측정 특성 13개 + 이 실습에서 파생한 범주 특성 2개
  - 파생 범주 2개 = alcohol, magnesium을 구간으로 나눈 열
    (숫자처럼 보이지만 실제로는 범주)

- **target (이진분류)**: 1번 품종이면 1(양성), 나머지 두 품종은 0(음성)
  - 양성 59개 / 전체 178개 ≈ 33% → 약간 불균형

- **범주 경계 (고정값)**:
  - alcohol: 12.5, 13.5를 경계로 3구간 (low / middle / high)
  - magnesium: 88, 105를 경계로 3구간 (low / middle / high)
  - ※ 데이터를 보고 정하는 게 아니라, 미리 정해진 숫자로 고정

- **데이터 분할**:
  - 계층화(stratify)하여 개발:test = 80:20
  - 개발 데이터 안에서 다시 5-fold StratifiedKFold 교차검증

- **누수 방지 (중요)**:
  - 구간 경계를 "전체 데이터의 분위수"로 계산하면, 그 안에 든 val·test의
    분포를 훔쳐보게 되어 데이터 누수가 생긴다.
  - 그래서 경계를 미리 고정된 숫자로 박아, val·test 정보가 경계 결정에
    개입하지 못하게 막는다. (전처리 단계에서도 test는 성역)

- **평가 지표**:
  - 선택 기준: 평균 AP
  - 함께 확인: F1, ROC-AUC, 그리고 AP의 fold 표준편차

In [5]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)

from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)

In [6]:
SEED = 42

wine = load_wine(as_frame=True)
X = wine.data.copy()
# "품종 0이냐, 아니냐"의 이진분류로 변환. one-vs-rest 방식
# 첫번째 품종이 관심 대상
y = (wine.target == 0).astype(int)

# X.columns
# ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']

# X["alcohol"].describe()
# count    178.000000
# mean      13.000618
# std        0.811827
# min       11.030000
# 25%       12.362500
# 50%       13.050000
# 75%       13.677500
# max       14.830000

X["alcohol_band"] = pd.cut(
    X["alcohol"],
    bins=[-np.inf, 12.5, 13.5, np.inf], # 12.5 이하, 12.5 초과 ~ 13.5 이하, 13.5 초과
    labels=["low", "middle", "high"],
).astype("category")

# X["magnesium"].describe()
# count    178.000000
# mean      99.741573
# std       14.282484
# min       70.000000
# 25%       88.000000
# 50%       98.000000
# 75%      107.000000
# max      162.000000

X["magnesium_band"] = pd.cut(
    X["magnesium"],
    bins=[-np.inf, 88, 105, np.inf], # 88 이하, 88 초과 ~ 105 이하, 105 초과
    labels=["low", "middle", "high" ],
).astype("category")

cat_cols = ["alcohol_band", "magnesium_band"]
num_cols = [column for column in X.columns if column not in cat_cols]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

cv = StratifiedKFold(
    # n_splits: Int = 5,
    n_splits=5,
    # *,
    # shuffle: bool = False,
    shuffle=True,
    # random_state: Int | RandomState | None = None
    random_state=SEED,
)

In [7]:
def make_categorical_preprocessor():
    """
    범주형 변수 전용 전처리 파이프라인을 생성

    1. 결측값을 해당 열에서 가장 자주 등장한 값으로 대체
    2. 범주형 값을 One-Hot Encoding하여 수치형 데이터로 변환
    """
    return Pipeline(
        [
            ("impute", SimpleImputer(
                # *,
                # missing_values: float | str | None = ...,
                # strategy: str = "mean",
                strategy="most_frequent",
                # fill_value: float | str | None = None,
                # verbose: Int | str = "deprecated",
                # copy: bool = True,
                # add_indicator: bool = False,
                # keep_empty_features: bool = False
            )),
            ("onehot", OneHotEncoder(
                # *,
                # categories: Sequence[ArrayLike] | Literal['auto'] = "auto",
                # drop: ArrayLike | None = None,
                # sparse: str | bool = "deprecated",
                # sparse_output: bool = True,
                sparse_output=False, # 0이든 1이든 모든 칸을 다 저장하는 평범한 배열
                # dtype: ... = ...,
                # handle_unknown: Literal['error', 'ignore', 'infrequent_if_exist'] = "error",
                handle_unknown="ignore",
                # min_frequency: float | None = None,
                # max_categories: Int | None = None
            ))
        ]
    )

def make_preprocessor(num_cols, cat_cols):
    """
    수치형 변수와 범주형 변수에 서로 다른 전처리를 적용하는 전체 전처리기를 생성

    - 수치형 변수: 결측값을 중앙값으로 대체
    - 범주형 변수: 결측값 대체 + One-Hot Encoding
    """    
    return ColumnTransformer(
        # transformers: Sequence[tuple],
        [
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", make_categorical_preprocessor(), cat_cols),
        ],
        # *,
        # remainder: BaseEstimator | Literal['drop', 'passthrough'] = "drop",
        # sparse_threshold: Float = 0.3,
        sparse_threshold=0.0, # 밀도(density, non-zero 비율)가 임계값 보다 작으면 희소행렬로, 크면  밀집배열로 출력
        # n_jobs: Int | None = None,
        # transformer_weights: dict | None = None,
        # verbose: bool = False,
        # verbose_feature_names_out: bool = True
    )

def make_pipeline(model, num_cols, cat_cols):
    """
    전처리 과정과 머신러닝 모델을 하나의 Pipeline으로 연결

    fit() 시:
        원본 데이터 → 전처리 학습/변환 → 모델 학습

    predict() 시:
        원본 데이터 → 동일한 전처리 적용 → 모델 예측

    전처리를 Pipeline 안에 포함시키므로 Cross Validation을 수행할 때 각 fold의 학습 데이터만 사용하여 결측값 대체 기준과 범주를 학습
    """
    return Pipeline(
        [
            ("pre", make_preprocessor(num_cols, cat_cols)),
            ("model", model)
        ]
    )

def evaluate(name, pipeline, X, y, cv):
    """하나의 후보 파이프라인을 공통 fold/지표로 교차검증하고 요약 통계를 반환"""
    scoring = {"AP": "average_precision", "ROC_AUC": "roc_auc", "F1": "f1"}
    cv_res = cross_validate(
        # estimator: BaseEstimator,
        pipeline,
        # X: MatrixLike,
        # y: MatrixLike | ArrayLike | None = None,
        X, y,
        # *,
        # groups: ArrayLike | None = None,
        # scoring: ArrayLike | tuple | ((...) -> Any) | Mapping | None = None,
        scoring=scoring,
        # cv: int | BaseCrossValidator | Iterable | None = None,
        cv=cv,
        # n_jobs: Int | None = None,
        n_jobs=-1,
        # verbose: Int = 0,
        # fit_params: dict | None = None,
        # pre_dispatch: Int | str = "2*n_jobs",
        # return_train_score: bool = False, # True 인 경우 학습한 데이터로 평가 하며 과적합 판단을 위해 비교용으로 사용
        # return_estimator: bool = False,
        # error_score: Float | str = np.nan
    )
    return {
        # cross_validate는 "test_" + 키이름으로 결과 키를 생성
        "candidate":    name,
        "AP_mean":      cv_res["test_AP"].mean(),
        "AP_std":       cv_res["test_AP"].std(),
        "F1_mean":      cv_res["test_F1"].mean(),
        "ROC_AUC_mean": cv_res["test_ROC_AUC"].mean(),   
    }

### 문제 1-1: 세 앙상블 모델을 같은 조건에서 비교하기
- Random Forest, AdaBoost, Gradient Boosting을 dev 데이터에서 나란히 비교한다.
- 순수하게 모델 차이만 드러나도록 전처리·fold·지표는 셋 다 동일하게 맞춘다.
- 파생 범주 두 개(`alcohol_band`, `magnesium_band`)가 실제로 one-hot 특성으로 펼쳐졌는지 변환된 특성 이름으로 확인한다.

### 수행해야 할 작업
1. 세 모델을 각각 `make_pipeline()`으로 감싸 후보 사전(dict)을 구성한다.
2. 공통 `evaluate()` 함수로 후보들을 같은 방식으로 평가한다.
3. 평균 AP 내림차순 비교표를 만들고, AP 표준편차·F1·ROC-AUC를 함께 표시한다.
4. Gradient Boosting Pipeline 하나를 dev 데이터에 적합(fit)한다.
5. 변환된 특성 이름에 `alcohol_band`와 `magnesium_band`가 모두 들어 있는지 assertion으로 확인한다.
6. 지금 데이터와 설정에서 어떤 후보를 먼저 검토할지 한 문장으로 정리한다.

In [17]:
def compare_ensemble_models(X_dev, y_dev, cv):
    """세 후보의 cv 표와 후보 Pipeline 사전을 반환"""
    templates = {
        "RandomForest": make_pipeline(RandomForestClassifier(random_state=SEED), num_cols, cat_cols),
        "AdaBoost": make_pipeline(AdaBoostClassifier(random_state=SEED), num_cols, cat_cols),
        "GradientBoosting": make_pipeline(GradientBoostingClassifier(random_state=SEED), num_cols, cat_cols)
    }

    rows = [evaluate(name, pipe, X_dev, y_dev, cv) for name, pipe in templates.items()]
    model_table = pd.DataFrame(rows).sort_values(by="AP_mean", ascending=False)

    templates["GradientBoosting"].fit(X_dev, y_dev)
    feature_names = templates["GradientBoosting"].named_steps["pre"].get_feature_names_out()
    # print(feature_names)
    assert any("alcohol_band" in name for name in feature_names)
    assert any("magnesium_band" in name for name in feature_names)

    return model_table, templates

In [20]:
model_table, templates = compare_ensemble_models(X_dev, y_dev, cv)
print(model_table.to_markdown())

|    | candidate        |   AP_mean |     AP_std |   F1_mean |   ROC_AUC_mean |
|---:|:-----------------|----------:|-----------:|----------:|---------------:|
|  0 | RandomForest     |  1        | 0          |  0.988235 |       1        |
|  1 | AdaBoost         |  0.995556 | 0.00544331 |  0.977709 |       0.998246 |
|  2 | GradientBoosting |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |


|    | candidate        |   AP_mean |     AP_std |   F1_mean |   ROC_AUC_mean |
|---:|:-----------------|----------:|-----------:|----------:|---------------:|
|  0 | RandomForest     |  1        | 0          |  0.988235 |       1        |
|  1 | AdaBoost         |  0.995556 | 0.00544331 |  0.977709 |       0.998246 |
|  2 | GradientBoosting |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |

- RF가 평균 AP가 가장 높고 fold 간 표준편차도 가장 작아 우선 검토 후보로 둔다.
- 다만, 해당 지표는 AdaBoost와 GradientBoosting이 기본 설정(학습률과 반복 수)에서 나온 결과일 뿐, 조율하면 결과는 달라질 수 있다. 

### 문제 2-1: 학습률과 트리 수의 조합 비교하기
- Gradient Boosting에서 학습률이 작을수록 항상 더 좋은지 확인한다.
- 학습률을 낮추면 한 단계에서 오차를 보정하는 폭이 작아져 더 많은 트리가 필요할 수 있지만, 트리를 늘린다고 validation 성능이 반드시 오르지는 않는다.

### 수행해야 할 작업
1. `(learning_rate, n_estimators)` 조합 `(0.10, 80)`, `(0.05, 160)`, `(0.02, 300)`을 준비한다.
2. 문제 1과 동일한 전처리·fold·AP 기준으로 각 조합을 평가한다.
3. 각 조합의 후보를 `templates`에 등록해 심화 문제에서 다시 사용할 수 있게 한다.
4. 평균 AP·fold 표준편차·학습률·트리 수를 한 표에 정리한다.
5. 모든 AP 값이 0과 1 사이에 있는지 확인한다.
6. 성능과 트리 수를 함께 보고 어떤 조합을 우선 검토할지 설명한다.

나무 1: 데이터를 보고 대충 예측을 냅니다. 아직 목표와 차이(오차)가 크게 남아요.

나무 2: 나무 1을 고치는 게 아니라, 나무 1이 남긴 오차를 새로 들여다봅니다. "지금 얼마나 틀렸지?"를 입력으로 받아서, 그 오차를 예측하는 자기만의 새 나무를 만들어요. 그 결과를 나무 1의 예측에 더합니다.

나무 3: 나무 1+2를 합친 뒤 아직도 남은 오차를 봅니다. 그걸 예측하는 또 다른 새 나무를 만들어 더합니다.

최종 예측 = 나무1의 예측
          + (학습률 × 나무2의 예측)
          + (학습률 × 나무3의 예측)
          + ...

predict를 부르면 내부에서 이 100그루를 전부 통과시켜 값을 누적한 뒤 결과를 냅니다.

In [25]:
config_rows = []
for learning_rate, n_estimators in [(0.1, 80), (0.05, 160), (0.02, 300)]:
    # for
    name = f"GB_lr{learning_rate}_n{n_estimators}"
    # for
    template = make_pipeline(
        GradientBoostingClassifier(
            # *,
            # loss: Literal['log_loss', 'deviance', 'exponential'] = "log_loss",
            # learning_rate: Float = 0.1,
            learning_rate=learning_rate,
            # n_estimators: Int = 100,
            n_estimators=n_estimators,
            # subsample: Float = 1,
            # criterion: Literal['friedman_mse', 'squared_error'] = "friedman_mse",
            # min_samples_split: float = 2,
            # min_samples_leaf: float = 1,
            # min_weight_fraction_leaf: Float = 0,
            # max_depth: int | None = 3,
            # min_impurity_decrease: Float = 0,
            # init: str | BaseEstimator | None = None,
            # random_state: Int | RandomState | None = None,
            random_state=SEED,
            # max_features: float | Literal['auto', 'sqrt', 'log2'] | None = None,
            # verbose: Int = 0,
            # max_leaf_nodes: Int | None = None,
            # warm_start: bool = False,
            # validation_fraction: Float = 0.1,
            # n_iter_no_change: Int | None = None,
            # tol: Float = 0.0001,
            # ccp_alpha: float = 0
        ),
        num_cols,
        cat_cols,
    )
    # for
    templates[name] = template
    # for
    row = evaluate(name, template, X_dev, y_dev, cv)
    row.update(
        {"learning_rate": learning_rate, "n_estimators": n_estimators}
    )
    # for
    config_rows.append(row)

config_table = pd.DataFrame(config_rows)[
    ["candidate", "learning_rate", "n_estimators", "AP_mean", "AP_std", "F1_mean", "ROC_AUC_mean"]
]
config_table = config_table.sort_values(by="AP_mean", ascending=False)
print(config_table.to_markdown(index=False))

assert config_table["AP_mean"].between(0.0, 1.0).all()


| candidate      |   learning_rate |   n_estimators |   AP_mean |    AP_std |   F1_mean |   ROC_AUC_mean |
|:---------------|----------------:|---------------:|----------:|----------:|----------:|---------------:|
| GB_lr0.1_n80   |            0.1  |             80 |  0.910718 | 0.0581066 |   0.91009 |       0.953684 |
| GB_lr0.05_n160 |            0.05 |            160 |  0.910454 | 0.0580398 |   0.91009 |       0.953099 |
| GB_lr0.02_n300 |            0.02 |            300 |  0.907486 | 0.063375  |   0.91009 |       0.945731 |


| candidate      |   learning_rate |   n_estimators |   AP_mean |    AP_std |   F1_mean |   ROC_AUC_mean |
|:---------------|----------------:|---------------:|----------:|----------:|----------:|---------------:|
| GB_lr0.1_n80   |            0.1  |             80 |  0.910718 | 0.0581066 |   0.91009 |       0.953684 |
| GB_lr0.05_n160 |            0.05 |            160 |  0.910454 | 0.0580398 |   0.91009 |       0.953099 |
| GB_lr0.02_n300 |            0.02 |            300 |  0.907486 | 0.063375  |   0.91009 |       0.945731 |

- 세 조합의 평균 AP 차이(약 0.003)가 fold 표준편차(약 0.058)보다 훨씬 작아 사실상 구분되지 않으므로,
- 학습률을 낮추고 트리를 늘린 이점은 없다.
- 따라서 가장 단순하고 트리 수가 적은 GB_lr0.1_n80을 우선 검토 후보로 둔다.

### 문제 3-1: Early Stopping을 진단하고 최종 후보 고정하기
- HistGradientBoosting이 개발 데이터 내부의 validation을 이용해 몇 번째 반복에서 학습을 멈추는지 확인한다.
- 이 값은 반복 수를 진단하기 위한 별도 holdout 실험일 뿐, 후보 순위를 매기는 근거가 아니다.
- 최종 배포 후보는 문제 1·2에서 같은 5-fold CV로 비교한 모델 가운데 평균 AP가 가장 높은 객체로 고정한다.

### 수행해야 할 작업
1. 개발 데이터를 fit:validation=`80:20`으로 계층화하여 한 번 더 분할한다.
2. `max_iter=500`, `learning_rate=0.05`, `n_iter_no_change=20`으로 HistGradientBoosting에 Early Stopping을 적용해 학습시킨다.
3. 실제로 멈춘 반복 수(`n_iter_`)와 holdout에서의 AP·F1·ROC-AUC를 출력한다.
4. `model_table`과 `config_table`을 합쳐, 평균 AP가 가장 높은 후보의 이름을 찾는다.
5. 그 이름으로 `templates`에서 후보 객체를 꺼내 개발 데이터 전체로 학습시킨다.
6. test에서 AP·F1·ROC-AUC를 한 번만 평가하고, 선택된 이름이 CV 1위와 같은지 확인한다.

- 트리를 한 그루씩 추가하면서 "성능이 더 안 오르면 멈춘다".
- 그런데 과연 이 "성능"을 어디서 재야 할까?
- 바로 test가 아닌 학습에 쓰지 않은 별도의 조각에서 성능을 재야 한다.

CV의 valid 조각 — 모델이 80그루를 다 만든 뒤에, 그 완성품을 채점. 학습 과정에 개입 안 함. "다 만들어진 모델을 평가"하는 사후 채점.

Early Stopping의 validation 조각 — 트리를 한 그루 추가할 때마다 실시간으로 채점해서, "더 만들까 말까"를 학습 도중에 결정. 학습 과정 안에 박혀 있음.

In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    # *arrays: Any,
    X_dev, y_dev,
    # test_size: Float | None = None,
    test_size = 0.2,
    # train_size: Float | None = None,
    # random_state: Int | RandomState | None = None,
    random_state = SEED,
    # shuffle: bool = True,
    # stratify: ArrayLike | None = None
    stratify = y_dev,
)

hist_gb_pipe = make_pipeline(
    HistGradientBoostingClassifier(
        # loss: Literal['log_loss', 'auto', 'binary_crossentropy', 'categorical_crossentropy'] = "log_loss",
        # *,
        # learning_rate: Float = 0.1,
        learning_rate = 0.05,
        # max_iter: Int = 100,
        max_iter = 500,
        # max_leaf_nodes: int | None = 31,
        # max_depth: int | None = None,
        # min_samples_leaf: Int = 20,
        # l2_regularization: Float = 0,
        # max_features: Float = 1,
        # max_bins: Int = 255,
        # categorical_features: MatrixLike | ArrayLike | None = None,
        # monotonic_cst: ArrayLike | Mapping | None = None,
        # interaction_cst: Sequence[list[int] | tuple[int, ...] | set[int]] | Literal['pairwise', 'no_interaction'] | None = None,
        # warm_start: bool = False,
        # early_stopping: bool | Literal['auto'] = "auto",
        early_stopping = True, # "auto" → 샘플이 10,000개보다 많으면 켜고, 적으면 끈다.
        # scoring: str | ((...) -> Any) | None = "loss",
        # validation_fraction: float | None = 0.1,
        validation_fraction = 0.15,
        # n_iter_no_change: Int = 10,
        n_iter_no_change = 20,
        # tol: Float = 1e-7,
        # verbose: Int = 0,
        # random_state: Int | RandomState | None = None,
        random_state = SEED,
        # class_weight: Mapping | str | None = None
    ), num_cols, cat_cols
)
hist_gb_pipe.fit(X_fit, y_fit)

hist_gb_proba = hist_gb_pipe.predict_proba(X_valid)[:, 1]

# "홀드아웃에서 대략 말이 되는 성능을 내는가"를 눈으로 확인
hist_gb_result = {
    "n_iter": hist_gb_pipe.named_steps["model"].n_iter_,
    "AP": average_precision_score(y_valid, hist_gb_proba),
    "F1": f1_score(y_valid, (hist_gb_proba >= 0.5).astype(int)),
    "ROC_AUC": roc_auc_score(y_valid, hist_gb_proba),
}
hist_gb_df = pd.DataFrame([hist_gb_result])
# print(hist_gb_df.to_markdown(index=False))

# print(hist_gb_df.to_markdown(index=False))
# {'n_iter': 393, 'AP': 0.9999999999999999, 'F1': 1.0, 'ROC_AUC': 1.0}

cv_results = pd.concat([model_table, config_table])
cv_results = cv_results.sort_values(by="AP_mean", ascending=False)
print(cv_results.to_markdown(index=False))

selected_candidate = cv_results.sort_values("AP_mean", ascending=False).iloc[0]["candidate"]
print(selected_candidate)

| candidate        |   AP_mean |     AP_std |   F1_mean |   ROC_AUC_mean |   learning_rate |   n_estimators |
|:-----------------|----------:|-----------:|----------:|---------------:|----------------:|---------------:|
| RandomForest     |  1        | 0          |  0.988235 |       1        |          nan    |            nan |
| AdaBoost         |  0.995556 | 0.00544331 |  0.977709 |       0.998246 |          nan    |            nan |
| GradientBoosting |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |          nan    |            nan |
| GB_lr0.1_n80     |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |            0.1  |             80 |
| GB_lr0.05_n160   |  0.910454 | 0.0580398  |  0.91009  |       0.953099 |            0.05 |            160 |
| GB_lr0.02_n300   |  0.907486 | 0.063375   |  0.91009  |       0.945731 |            0.02 |            300 |
RandomForest


In [51]:
# print(list(templates.keys()))
# ['RandomForest', 'AdaBoost', 'GradientBoosting', 'GB_lr0.1_n80', 'GB_lr0.05_n160', 'GB_lr0.02_n300']

# print(selected_candidate) # RandomForest
final_template = templates[selected_candidate]
final_template.fit(X_dev, y_dev)

test_proba = final_template.predict_proba(X_test)[:, 1]

test_AP = average_precision_score(y_test, test_proba)
test_roc_auc = roc_auc_score(y_test, (test_proba >= 0.5).astype(int))
test_f1 = f1_score(y_test, (test_proba >= 0.5).astype(int))

print(f"test_AP: {test_AP}")
print(f"test_ROC_AUC: {test_roc_auc}")
print(f"test_f1: {test_f1}")

test_AP: 0.9935897435897436
test_ROC_AUC: 0.9791666666666667
test_f1: 0.96


### 제출해야 할 보고 형식
#### Early Stopping의 실제 반복 수와 별도 validation 지표 3개

|   n_iter |   AP |   F1 |   ROC_AUC |
|---------:|-----:|-----:|----------:|
|      393 |    1 |    1 |         1 |
---
#### 동일 CV 후보들의 최종 선택 이름

| candidate        |   AP_mean |     AP_std |   F1_mean |   ROC_AUC_mean |   learning_rate |   n_estimators |
|:-----------------|----------:|-----------:|----------:|---------------:|----------------:|---------------:|
| RandomForest     |  1        | 0          |  0.988235 |       1        |          nan    |            nan |
| AdaBoost         |  0.995556 | 0.00544331 |  0.977709 |       0.998246 |          nan    |            nan |
| GradientBoosting |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |          nan    |            nan |
| GB_lr0.1_n80     |  0.910718 | 0.0581066  |  0.91009  |       0.953684 |            0.1  |             80 |
| GB_lr0.05_n160   |  0.910454 | 0.0580398  |  0.91009  |       0.953099 |            0.05 |            160 |
| GB_lr0.02_n300   |  0.907486 | 0.063375   |  0.91009  |       0.945731 |            0.02 |            300 |
---
#### 선택된 fitted 모델의 test AP·F1·ROC-AUC
- test_AP: 0.9935897435897436
- test_f1: 0.96
- test_ROC_AUC: 0.9791666666666667

---
#### Early Stopping holdout 점수를 CV 순위표에 직접 섞지 않았다는 확인
- Early Stopping의 역할은 n_iter 진단에 한정했고, 최종 후보는 오직 CV 평균 AP 1위인 RandomForest로 고정했다.